# Step 5 - Define and Identify the Liquidity Crossover

Primary target: the first observed day when next-contract volume share is strictly greater than 50%. Open-interest crossover is recorded independently as a secondary signal. A crossover is a measurement rule; it does not mean every trader rolls on that date. The first-crossing rule does not require the share to stay above 50% afterward.

In [1]:
import pandas as pd
import numpy as np

panel = pd.read_parquet('../data/processed/zn_roll_panel.parquet')
panel['date'] = pd.to_datetime(panel['date'])
panel = panel.sort_values(['roll_id', 'date']).copy()
required = {'roll_id', 'front', 'next', 'date', 'bdays_to_fnd', 'next_volume_share', 'next_oi_share'}
missing = required - set(panel.columns)
if missing:
    raise ValueError(f'Missing panel columns: {sorted(missing)}')
VOLUME_THRESHOLD = 0.50
OI_THRESHOLD = 0.50
print(f'Loaded {panel.roll_id.nunique()} rolls and {len(panel):,} panel rows.')

Loaded 27 rolls and 837 panel rows.


## Event rules

- Primary outcome: earliest observed date with `next_volume_share > 0.50`.
- Secondary signal: earliest observed date with `next_oi_share > 0.50`, calculated separately.
- If no qualifying observation exists in the panel window, the date and relative day are missing and the observed flag is false.
- A share exactly equal to 0.50 does not cross; missing shares are ignored.

In [2]:
events = []
for roll_id, grp in panel.groupby('roll_id', sort=True):
    grp = grp.sort_values('date')
    volume_hits = grp.loc[grp['next_volume_share'] > VOLUME_THRESHOLD]
    oi_hits = grp.loc[grp['next_oi_share'] > OI_THRESHOLD]
    volume_event = volume_hits.iloc[0] if not volume_hits.empty else None
    oi_event = oi_hits.iloc[0] if not oi_hits.empty else None
    first = grp.iloc[0]
    events.append({
        'roll_id': roll_id,
        'front': first['front'],
        'next': first['next'],
        'volume_crossover_observed': volume_event is not None,
        'volume_crossover_date': volume_event['date'] if volume_event is not None else pd.NaT,
        'volume_crossover_bdays_to_fnd': volume_event['bdays_to_fnd'] if volume_event is not None else np.nan,
        'oi_crossover_observed': oi_event is not None,
        'oi_crossover_date': oi_event['date'] if oi_event is not None else pd.NaT,
        'oi_crossover_bdays_to_fnd': oi_event['bdays_to_fnd'] if oi_event is not None else np.nan,
        'volume_threshold': VOLUME_THRESHOLD,
        'oi_threshold': OI_THRESHOLD,
    })
events = pd.DataFrame(events).sort_values('roll_id').reset_index(drop=True)
print(f"Volume crossovers: {events['volume_crossover_observed'].sum()} / {len(events)} rolls")
print(f"OI crossovers: {events['oi_crossover_observed'].sum()} / {len(events)} rolls")
display(events)

Volume crossovers: 27 / 27 rolls
OI crossovers: 27 / 27 rolls


,roll_id,front,next,volume_crossover_observed,volume_crossover_date,volume_crossover_bdays_to_fnd,oi_crossover_observed,oi_crossover_date,oi_crossover_bdays_to_fnd,volume_threshold,oi_threshold
0,ZNH0->ZNM0,ZNH0,ZNM0,True,2020-02-27,-1,True,2020-02-26,-2,0.5,0.5
1,ZNH1->ZNM1,ZNH1,ZNM1,True,2021-02-25,-1,True,2021-02-24,-2,0.5,0.5
2,ZNH2->ZNM2,ZNH2,ZNM2,True,2022-02-25,-2,True,2022-02-24,-3,0.5,0.5
3,ZNH3->ZNM3,ZNH3,ZNM3,True,2023-02-26,-2,True,2023-02-23,-4,0.5,0.5
4,ZNH4->ZNM4,ZNH4,ZNM4,True,2024-02-28,-1,True,2024-02-25,-4,0.5,0.5
5,ZNH5->ZNM5,ZNH5,ZNM5,True,2025-02-26,-2,True,2025-02-25,-3,0.5,0.5
6,ZNH6->ZNM6,ZNH6,ZNM6,True,2026-02-25,-2,True,2026-02-25,-2,0.5,0.5
7,ZNM0->ZNU0,ZNM0,ZNU0,True,2020-05-28,-1,True,2020-05-27,-2,0.5,0.5
8,ZNM1->ZNU1,ZNM1,ZNU1,True,2021-05-27,-3,True,2021-05-26,-4,0.5,0.5
9,ZNM2->ZNU2,ZNM2,ZNU2,True,2022-05-27,-3,True,2022-05-26,-4,0.5,0.5


## Timing summary

Negative values mean the event occurred before FND. These are descriptions of observed events, not out-of-sample forecasts.

In [3]:
timing_summary = pd.DataFrame({
    'volume_crossover_bdays_to_fnd': events['volume_crossover_bdays_to_fnd'].describe(),
    'oi_crossover_bdays_to_fnd': events['oi_crossover_bdays_to_fnd'].describe(),
}).T
display(timing_summary)
paired = events.dropna(subset=['volume_crossover_bdays_to_fnd', 'oi_crossover_bdays_to_fnd']).copy()
paired['oi_minus_volume_bdays'] = (paired['oi_crossover_bdays_to_fnd'] - paired['volume_crossover_bdays_to_fnd'])
print(f'Rolls with both events: {len(paired)}')
if not paired.empty:
    print('Median (OI timing - volume timing), business days:', paired['oi_minus_volume_bdays'].median())
display(paired[['roll_id', 'volume_crossover_bdays_to_fnd', 'oi_crossover_bdays_to_fnd', 'oi_minus_volume_bdays']])

,count,mean,std,min,25%,50%,75%,max
volume_crossover_bdays_to_fnd,27.0,-2.037037,0.587137,-3.0,-2.0,-2.0,-2.0,-1.0
oi_crossover_bdays_to_fnd,27.0,-3.444444,1.050031,-5.0,-4.0,-3.0,-3.0,-2.0


Rolls with both events: 27
Median (OI timing - volume timing), business days: -1.0


,roll_id,volume_crossover_bdays_to_fnd,oi_crossover_bdays_to_fnd,oi_minus_volume_bdays
0,ZNH0->ZNM0,-1,-2,-1
1,ZNH1->ZNM1,-1,-2,-1
2,ZNH2->ZNM2,-2,-3,-1
3,ZNH3->ZNM3,-2,-4,-2
4,ZNH4->ZNM4,-1,-4,-3
5,ZNH5->ZNM5,-2,-3,-1
6,ZNH6->ZNM6,-2,-2,0
7,ZNM0->ZNU0,-1,-2,-1
8,ZNM1->ZNU1,-3,-4,-1
9,ZNM2->ZNU2,-3,-4,-1


## Save roll-level target table

The output contains one row per roll and can be joined back to the daily panel when building features for prediction.

In [4]:
from pathlib import Path

output_path = Path('../data/processed/zn_crossover_events.parquet')
output_path.parent.mkdir(parents=True, exist_ok=True)
events.to_parquet(output_path, index=False)
print(f'Saved {len(events)} roll-level event rows to {output_path}')

Saved 27 roll-level event rows to ..\data\processed\zn_crossover_events.parquet
